In [3]:
!pip install pyspark -q

In [ ]:
Base_path = "/content/drive/MyDrive/Ride_Sharing_Project"

Raw_path = f"{Base_path}/Raw_Data"
Bronze_path = f"{Base_path}/Bronze_Data"
Silver_path = f"{Base_path}/Silver_Data"
Gold_path = f"{Base_path}/Gold_Data"

In [6]:
drivers_Path = f"{Raw_path}/drivers.csv"
trips_Path = f"{Raw_path}/trips.csv"
trips_logs = f"{Raw_path}/trips_logs.csv"

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder\
.appName("Ride_Sharing_Project") \
.getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.3


In [8]:
drivers_schema = StructType([
    StructField("driver_id", IntegerType() , True),
    StructField("name", StringType(), True),
    StructField("city", StringType() , True),
    StructField("rating" , FloatType(), True)
])

trips_schema = StructType([
    StructField("trip_id" , IntegerType(), True),
    StructField("driver_id" , IntegerType(), True),
    StructField("pickup_location" , StringType(), True),
    StructField("drop_location" , StringType(), True),
    StructField("distance_km" , FloatType(), True),
    StructField("fare_amount" , FloatType(), True),
    StructField("trip_status" , StringType(), True),
])

trips_logs_schema = StructType([
    StructField("log_id" , IntegerType(), True),
    StructField("trip_id" , IntegerType(), True),
    StructField("start_time" , TimestampType(), True),
    StructField("end_time" , TimestampType(), True),
    StructField("delay_minutes" , FloatType(), True),
    StructField("cancellation_flag" , IntegerType(), True),
])

In [9]:
drivers_df = spark.read \
    .option("header", True) \
    .schema(drivers_schema) \
    .csv(f"{Raw_path}/drivers.csv")


trips_df = spark.read\
.option("header" , True) \
.schema(trips_schema) \
.csv(f"{Raw_path}/trips.csv")

trips_logs_df = spark.read\
.option("header" , True) \
.option("timestampFormat" , "yyyy-MM-dd HH:mm:ss") \
.schema(trips_logs_schema) \
.csv(f"{Raw_path}/trip_logs.csv")

In [10]:
print("Drivers rows:" , drivers_df.count())
print("Trips rows:" ,trips_df.count())
print("Trip Logs rows:" ,trips_logs_df.count())

drivers_df.show(5)
trips_df.show(5)
trips_logs_df.show(5)

Drivers rows: 150
Trips rows: 150
Trip Logs rows: 150
+---------+-------+------+------+
|driver_id|   name|  city|rating|
+---------+-------+------+------+
|        1|Rahul_1| Delhi|   4.6|
|        2|Priya_2|Mumbai|   3.7|
|        3|Rahul_3|  Pune|   3.6|
|        4|Sneha_4| Delhi|   3.5|
|        5|Priya_5|Mumbai|   4.3|
+---------+-------+------+------+
only showing top 5 rows
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+-------------+-----------+-----------+-----------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed|
|      3|      132|Railway Station|         Mall|      17.27|     213.02|  Completed|
|      4|       58|Railway Station|      IT Park|      20.55|      185.6|  Completed|
|      5|     

In [11]:
drivers_df.write \
.mode("overwrite") \
.parquet(f"{Bronze_path}/drivers")

trips_df.write \
.mode("overwrite") \
.parquet(f"{Bronze_path}/trips")

trips_logs_df.write \
.mode("overwrite") \
.parquet(f"{Bronze_path}/trip_logs")

print("Bronze layer written successfully")

Bronze layer written successfully


In [12]:
bronze_drivers = spark.read.parquet(f"{Bronze_path}/drivers")
bronze_trips = spark.read.parquet(f"{Bronze_path}/trips")
bronze_trip_logs = spark.read.parquet(f"{Bronze_path}/trip_logs")

print("Bronze Drivers:", bronze_drivers.count())
print("Bronze Trips:", bronze_trips.count())
print("Bronze Trip Logs:", bronze_trip_logs.count())

Bronze Drivers: 150
Bronze Trips: 150
Bronze Trip Logs: 150


**SILVER LAYER**

In [13]:
silver_df = bronze_trips \
 .join(bronze_drivers , on = "driver_id" , how ="left")\
.join(bronze_trip_logs, on = "trip_id" , how="left")

print("Rows after join:" , silver_df.count())
silver_df.show(5)

Rows after join: 150
+-------+---------+---------------+-------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|trip_id|driver_id|pickup_location|drop_location|distance_km|fare_amount|trip_status|     name|     city|rating|log_id|         start_time|           end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+-------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|      1|       78|        IT Park|      IT Park|        8.6|        0.0|  Cancelled|Anjali_78|    Delhi|   5.0|     1|2025-01-03 01:44:00|               NULL|          0.0|                1|
|      2|      114|        Airport|         Mall|      17.54|     203.08|  Completed| Neha_114|Bangalore|   3.7|     2|2025-01-02 04:34:00|2025-01-02 04:50:00|         20.0|                0|
|      3|      132|

In [14]:
silver_df = silver_df.filter(
    (col("distance_km") > 0) &
    (col("fare_amount") >=0) &
    (col("start_time").isNotNull()) &
    (
        (col("trip_status") == "Cancelled") |
        (col("end_time").isNotNull())
    )
    )

In [15]:
silver_df = silver_df \
.withColumn("trip_duration_minutes" ,
            when(
                col("end_time").isNotNull(),
            (unix_timestamp("end_time") - unix_timestamp("start_time"))/60
            ).otherwise(None)

)\
.withColumn(
    "completion_flag" ,
    when(col("trip_status") == "Completed" ,1).otherwise(0)
) \
.withColumn(
    "cancelled_flag" ,
    when(col("trip_status" ) == "Cancelled" , 1).otherwise(0)
) \
.withColumn(
    "revenue_per_km",
    when(
        col("distance_km") >0 ,
        col("fare_amount") / col("distance_km")
    ).otherwise(None)
) \
.withColumn(
    "is_delayed" ,
    when(col("delay_minutes") >0,1).otherwise(0)
)


In [16]:
silver_df.write \
    .mode("overwrite") \
    .parquet(f"{Silver_path}/cleaned_trips")

print("Silver layer saved successfully")

Silver layer saved successfully


**GOLD LAYER**

In [17]:
driver_performance = silver_df\
.groupBy("driver_id" , "name" , "city" , "rating" ) \
.agg(
    count("trip_id").alias("total_trips"),
    sum("completion_flag").alias("completed_trips"),
    sum("cancelled_flag").alias("cancelled_trips"),
    round(avg("delay_minutes"),2).alias("total_delay_minutes"),
    round(sum("fare_amount") ,2).alias("total_fare_amount"),
    round(avg("revenue_per_km"),2).alias("avg_revenue_per_km")
) \
.withColumn(
    "completion_rate",
    round(col("completed_trips")/col("total_trips")*100,2)
) \
.withColumn(
    "cancellation_rate",
    round(col("cancelled_trips")/col("total_trips")*100,2)
)

driver_performance.show(10)

+---------+---------+---------+------+-----------+---------------+---------------+-------------------+-----------------+------------------+---------------+-----------------+
|driver_id|     name|     city|rating|total_trips|completed_trips|cancelled_trips|total_delay_minutes|total_fare_amount|avg_revenue_per_km|completion_rate|cancellation_rate|
+---------+---------+---------+------+-----------+---------------+---------------+-------------------+-----------------+------------------+---------------+-----------------+
|      124|Priya_124|   Mumbai|   5.0|          2|              0|              2|                0.0|              0.0|               0.0|            0.0|            100.0|
|      103|Vikas_103|Bangalore|   3.7|          3|              2|              1|               8.33|           256.07|              8.21|          66.67|            33.33|
|       65|  Amit_65|Bangalore|   3.6|          3|              3|              0|                9.0|           493.75|          

In [18]:
pickup_demand = silver_df\
.groupBy("pickup_location")\
.agg(
    count("trip_id").alias("total_rides"),
    sum("completion_flag").alias("completed_rides"),
    sum("cancelled_flag").alias("cancelled_rides"),
    round(sum("fare_amount"),2).alias("total_revenue")
) \
.orderBy(col("total_rides").desc())

pickup_demand.show()

+---------------+-----------+---------------+---------------+-------------+
|pickup_location|total_rides|completed_rides|cancelled_rides|total_revenue|
+---------------+-----------+---------------+---------------+-------------+
|        Airport|         36|             13|             23|      2075.39|
|        IT Park|         34|             17|             17|      3191.85|
|Railway Station|         29|             14|             15|      1722.94|
|           Mall|         26|             13|             13|      2339.98|
|    City Center|         25|              6|             19|       760.92|
+---------------+-----------+---------------+---------------+-------------+



In [19]:
revenue_analysis = silver_df\
.groupBy("city")\
.agg(
    round(sum("fare_amount"),2).alias("total revenue"),
    round(avg("fare_amount"),2).alias("avg fare"),
    round(avg("revenue_per_km"),2).alias("avg revenue per km"),
    count("trip_id").alias("total trips")
)\
.orderBy(col("total revenue").desc())

revenue_analysis.show()

+---------+-------------+--------+------------------+-----------+
|     city|total revenue|avg fare|avg revenue per km|total trips|
+---------+-------------+--------+------------------+-----------+
|    Delhi|      2851.14|   95.04|              7.15|         30|
|Bangalore|      2535.43|   68.53|              4.58|         37|
|   Mumbai|      2286.38|   71.45|              4.87|         32|
|     Pune|      1493.44|   43.92|              3.56|         34|
|Hyderabad|       924.69|   54.39|              5.01|         17|
+---------+-------------+--------+------------------+-----------+



In [20]:
delay_analysis = silver_df\
.groupBy("driver_id" , "name")\
.agg(
    round(avg("delay_minutes"),2).alias("avg delay minutes"),
    sum("is_delayed").alias("delayed trips"),
    count("trip_id").alias("total trips")
)\
.withColumn(
    "delay rate",
    round(col("delayed trips")/col("total trips")*100,2)
)\
.orderBy(col("avg delay minutes").desc())

delay_analysis.show(10)

+---------+----------+-----------------+-------------+-----------+----------+
|driver_id|      name|avg delay minutes|delayed trips|total trips|delay rate|
+---------+----------+-----------------+-------------+-----------+----------+
|       19|  Rahul_19|             20.0|            1|          1|     100.0|
|       99|  Rahul_99|             20.0|            1|          1|     100.0|
|      132| Priya_132|             19.0|            2|          2|     100.0|
|      143|Anjali_143|             19.0|            1|          1|     100.0|
|      137| Karan_137|             19.0|            1|          1|     100.0|
|       85|  Karan_85|             19.0|            1|          1|     100.0|
|       87|  Vikas_87|             18.0|            1|          1|     100.0|
|      111| Karan_111|             18.0|            1|          1|     100.0|
|      142| Sneha_142|             16.0|            1|          1|     100.0|
|       45| Anjali_45|             16.0|            1|          

In [21]:
driver_window = Window.orderBy(
    col("completion_rate").desc(),
    col("total_fare_amount").desc(),
    col("rating").desc()
)

driver_rankings = driver_performance \
    .withColumn(
        "driver rank",
        dense_rank().over(driver_window)
    )

driver_rankings.show(10)

+---------+---------+---------+------+-----------+---------------+---------------+-------------------+-----------------+------------------+---------------+-----------------+-----------+
|driver_id|     name|     city|rating|total_trips|completed_trips|cancelled_trips|total_delay_minutes|total_fare_amount|avg_revenue_per_km|completion_rate|cancellation_rate|driver rank|
+---------+---------+---------+------+-----------+---------------+---------------+-------------------+-----------------+------------------+---------------+-----------------+-----------+
|       65|  Amit_65|Bangalore|   3.6|          3|              3|              0|                9.0|           493.75|             11.31|          100.0|              0.0|          1|
|      102|Vikas_102|    Delhi|   4.8|          2|              2|              0|                7.0|           390.52|             12.59|          100.0|              0.0|          2|
|      132|Priya_132|Hyderabad|   4.0|          2|              2|    

In [22]:
driver_performance.write \
    .mode("overwrite") \
    .parquet(f"{Gold_path}/driver_performance")

pickup_demand.write \
    .mode("overwrite") \
    .parquet(f"{Gold_path}/pickup_demand")

revenue_analysis.write \
    .mode("overwrite") \
    .parquet(f"{Gold_path}/revenue_analysis")

delay_analysis.write \
    .mode("overwrite") \
    .parquet(f"{Gold_path}/delay_analysis")

driver_rankings.write \
    .mode("overwrite") \
    .parquet(f"{Gold_path}/driver_rankings")

print("Gold layer written successfully")

Gold layer written successfully


In [23]:
print("Driver Performance:", driver_performance.count())
print("Pickup Demand:", pickup_demand.count())
print("Revenue Analysis:", revenue_analysis.count())
print("Delay Analysis:", delay_analysis.count())
print("Driver Rankings:", driver_rankings.count())

Driver Performance: 99
Pickup Demand: 5
Revenue Analysis: 5
Delay Analysis: 99
Driver Rankings: 99


In [24]:
print("Bronze trips:" , bronze_trips.count())
print("Silver trips:" , silver_df.count())

print(
    "Invalid distance:",
    silver_df.filter(col("distance_km")<=0).count()
)
print(
    "Negative fare:",
    silver_df.filter(col("fare_amount")<0).count()
)

print(
    "Missing start time:",
    silver_df.filter(col("start_time").isNull()).count()
)
print(
    "Duplicate trips",
    silver_df.groupBy("trip_id").count().filter(col("count")>1).count()
)

Bronze trips: 150
Silver trips: 150
Invalid distance: 0
Negative fare: 0
Missing start time: 0
Duplicate trips 0


In [26]:
mismatch_count = silver_df.filter(
    ((col("trip_status")=="Cancelled") & (col("cancelled_flag") != 1)) |
    ((col("trip_status") == "Completed") & (col("cancelled_flag") !=0))
).count()

print("Cancellation status mismathces:" , mismatch_count)

Cancellation status mismathces: 0


In [28]:
silver_df.cache()
silver_df.count()

150

In [32]:
from pyspark.sql.functions import broadcast

silver_df = bronze_trips\
.join(
    broadcast(bronze_drivers),
    on="driver_id",
    how="left"
)\
.join(
    bronze_trip_logs,
    on="trip_id",
    how="left"
)